# Investing.com Feature Engineering

For delivery day D the forecast is made around 12:00 on D-1, when that day's market close is not known yet. The latest settled price is therefore from D-2, and every feature is shifted by 48 hours. `create_all_features` builds the lagged levels and the 7-day changes. Two selectors cut that frame into the tree set and the neural-network set.

**Tree features**

* `ttf_lag48`, `eua_lag48`, `coal_lag48` — price levels from D-2.
* `ttf_chg7d`, `eua_chg7d`, `coal_chg7d` — the percent change over 7 days, computed on the series already shifted by 48 hours.
* `de_baseload_lag48`, `de_baseload_chg7d` — the same level and 7-day change for German baseload.

**Neural-network features**

* `ttf_lag48`, `eua_lag48`, `coal_lag48`, `eurpln_lag48`, `de_baseload_lag48` — lagged price levels only. EUR/PLN is in this set and not in the tree set. The 7-day changes stay with the trees.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_merged_data,
    save_merged_data,
    inspect_merged_file,
)
from src.utils.feature_engineering.fe_investing import (
    create_all_features,
    select_tree_features,
    select_nn_features,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

data_inv = load_merged_data(
    DATA_DIR / "preprocessed" / "investing_com" / date_range / "investing_com.parquet"
)

## 1. Input Columns

The preprocessed prices, before the lag and the return features.

In [3]:
print(list(data_inv.columns))

['dutch_ttf_natural_gas_futures', 'eu_carbon_emissions_futures', 'eur_pln', 'german_power_baseload_futures', 'rotterdam_coal_futures']


## 2. Features for Tree Models

Lagged levels `ttf_lag48`, `eua_lag48`, `coal_lag48` and `de_baseload_lag48`. The 7-day changes `ttf_chg7d`, `eua_chg7d`, `coal_chg7d` and `de_baseload_chg7d` are the percent change over 168 hours of the series already shifted by 48 hours. EUR/PLN is not in this set.

In [4]:
features = create_all_features(data_inv)
data_tree = select_tree_features(features)

In [5]:
print(data_tree.shape)
print(list(data_tree.columns))

(33241, 8)
['ttf_lag48', 'eua_lag48', 'coal_lag48', 'ttf_chg7d', 'eua_chg7d', 'coal_chg7d', 'de_baseload_lag48', 'de_baseload_chg7d']


## 3. Features for Neural Networks

Lagged price levels only: `ttf_lag48`, `eua_lag48`, `coal_lag48`, `eurpln_lag48`, `de_baseload_lag48`. The 7-day changes stay in the tree set.

In [6]:
data_nn = select_nn_features(features)
print(data_nn.shape)
print(list(data_nn.columns))

(33241, 5)
['ttf_lag48', 'eua_lag48', 'coal_lag48', 'eurpln_lag48', 'de_baseload_lag48']


## 4. Conclusions

The decisions above are applied to the merged Investing.com frame. The full feature set and the neural-network subset are saved to `data/features/investing_com/trees/` and `data/features/investing_com/nn/`, and both files are verified by reading them back.

## 5. Saving Feature Data

In [7]:
trees_path = DATA_DIR / "features" / "investing_com" / "trees" / f"{date_range}.parquet"
nn_path = DATA_DIR / "features" / "investing_com" / "nn" / f"{date_range}.parquet"

save_merged_data(data_tree, trees_path)
save_merged_data(data_nn, nn_path)
inspect_merged_file(trees_path)
inspect_merged_file(nn_path)

  [FILE] data/features/investing_com/trees/2022-12-01_2026-09-16.parquet | shape: (33241, 8) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00 | cols: 8 | NaN: 1056
  [FILE] data/features/investing_com/nn/2022-12-01_2026-09-16.parquet | shape: (33241, 5) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00 | cols: 5 | NaN: 240
